In [1]:
from sqlalchemy import create_engine, URL
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()
url = URL.create(
    drivername="postgresql",
    username=os.environ.get("POSTGRES_USER"),
    password=Path("/run/secrets/postgres_password").read_text().strip(),
    host=os.environ.get("POSTGRES_HOST"),
    port=os.environ.get("POSTGRES_PORT"),
    database=os.environ.get("POSTGRES_DB")
)

engine = create_engine(url)

In [2]:
import pandas as pd
query = "SELECT * from employees"
df = pd.read_sql_query(query, engine)

In [3]:
from sqlalchemy import text

def fetch_employee_tuple(id):
    query = """
      SELECT * from employees
      WHERE emp_id = :id
      """

    with engine.connect() as conn:
        result = conn.execute(text(query),{'id':id}).one_or_none()

    return result

In [4]:
from sqlalchemy import text

def fetch_employee_dict(id):
    query = """
    SELECT * from employees
    WHERE emp_id = :id
    """

    with engine.connect() as conn:
        result = conn.execute(text(query),{'id':id}).mappings().one_or_none()

    return result

In [5]:
print(df[df['emp_id'] == 98])

    emp_id first_name last_name  experience                 skills
97      98    William     Moore         8.7  SQL, Machine Learning


In [6]:
print(fetch_employee_tuple(98))

(98, 'William', 'Moore', Decimal('8.7'), 'SQL, Machine Learning')


In [7]:
print(fetch_employee_dict(98))

{'emp_id': 98, 'first_name': 'William', 'last_name': 'Moore', 'experience': Decimal('8.7'), 'skills': 'SQL, Machine Learning'}


In [8]:
for k,v in fetch_employee_dict(98).items():
    print(k,":",v)

emp_id : 98
first_name : William
last_name : Moore
experience : 8.7
skills : SQL, Machine Learning
